In [1]:
#!/usr/bin/env python
# coding: utf-8
"""
智能合約處理系統 - 整合版
新增功能：
  - 問答介面內文件圖片預覽
  - QA 答案快取（儲存 / 語意檢索 / 管理刪除）
"""

import gradio as gr
from llama_index.core import Document, VectorStoreIndex
import os
import sys
import shutil
from pathlib import Path
from pdf2image import convert_from_path
import fitz  # PyMuPDF
import tempfile
import subprocess
import pdfplumber
import json
from PIL import Image
import ollama
import uuid
import datetime
import numpy as np

# ==========================================
# 0-A. 本地 Embedding 快取管理
# ==========================================

CACHE_MARKER = "docstore.json"
EMBEDDING_JSON_PATTERN = "*.json"


def _cache_dir(txt_path: str) -> Path:
    return Path(txt_path).parent


def _cache_marker(cache: Path) -> Path:
    return cache / CACHE_MARKER


def collection_exists_and_has_data(txt_path: str) -> bool:
    return _cache_marker(_cache_dir(txt_path)).exists()


def _get_cache_json_files(folder: Path) -> list:
    known = ["docstore.json", "index_store.json", "vector_store.json",
             "graph_store.json", "image__vector_store.json"]
    return [folder / f for f in known if (folder / f).exists()]


def list_cached_collections() -> list:
    result = []
    root = Path("./text_result")
    if not root.exists():
        return result
    for folder in root.iterdir():
        if not folder.is_dir():
            continue
        if _cache_marker(folder).exists():
            files = _get_cache_json_files(folder)
            size_mb = sum(f.stat().st_size for f in files) / 1024 / 1024
            result.append({
                "name": folder.name,
                "path": str(folder),
                "size_mb": round(size_mb, 2),
                "files": [f.name for f in files],
            })
    return result


def delete_cache_for_txt(txt_path: str) -> str:
    cache = _cache_dir(txt_path)
    files = _get_cache_json_files(cache)
    if not files:
        return f"⚠️ 找不到快取：{cache.name}"
    for f in files:
        f.unlink()
    names = [f.name for f in files]
    return f"✅ 已刪除快取（{cache.name}）：{', '.join(names)}"


def delete_all_cache() -> str:
    root = Path("./text_result")
    count = 0
    if root.exists():
        for folder in root.iterdir():
            if not folder.is_dir():
                continue
            for f in _get_cache_json_files(folder):
                f.unlink()
                count += 1
    return f"✅ 已清除所有 Embedding 快取（共刪除 {count} 個檔案）"


def build_or_load_index(txt_path: str, progress=None):
    from llama_index.core import StorageContext, load_index_from_storage

    if not RAG_AVAILABLE:
        return None, "❌ 系統缺少 RAG 組件，無法建立索引。"
    if not txt_path or not os.path.exists(txt_path):
        return None, "❌ 找不到文字檔案，請先執行 OCR。"

    cache = _cache_dir(txt_path)

    if collection_exists_and_has_data(txt_path):
        if progress:
            progress(0.2, desc="📂 發現本地快取，直接載入 Embedding...")
        try:
            storage_context = StorageContext.from_defaults(persist_dir=str(cache))
            index = load_index_from_storage(storage_context)
            if progress:
                progress(0.8, desc="🔗 組裝查詢引擎...")
            nodes = list(index.docstore.docs.values())
            engine = HierarchicalQueryEngine(index, nodes)
            if progress:
                progress(1.0, desc="✅ 完成！")
            return engine, f"✅ 已從本地快取載入（{len(nodes)} 個節點）：{cache.name}"
        except Exception as e:
            print(f"⚠️ 快取讀取失敗，將重新建立：{e}")
            for f in _get_cache_json_files(cache):
                f.unlink()

    if progress:
        progress(0.1, desc="📖 讀取文件內容...")
    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            text_content = f.read()
    except Exception as e:
        return None, f"❌ 讀取文件失敗：{e}"

    if not text_content.strip():
        return None, "❌ 文件內容為空"

    if progress:
        progress(0.2, desc="🔍 解析合約結構（階層式切割）...")

    doc = Document(
        text=text_content,
        metadata={
            "file_name":   Path(txt_path).name,
            "file_path":   txt_path,
            "contract_id": Path(txt_path).stem,
            "orig_path":   txt_path,
        },
    )

    parser = HierarchicalContractNodeParser()
    nodes  = parser.get_nodes_from_documents([doc])
    print(f"📊 解析完成，共 {len(nodes)} 個節點")

    if not nodes:
        return None, "❌ 文件解析失敗，未能提取任何節點"

    if progress:
        progress(0.4, desc=f"⚙️ 計算 Embedding（{len(nodes)} 個節點）...")

    try:
        import traceback as _tb
        from llama_index.core import StorageContext
        storage_context = StorageContext.from_defaults()
        index = VectorStoreIndex(nodes, storage_context=storage_context, show_progress=False)
    except BaseException as e:
        import traceback as _tb
        print("[ERROR] 建立向量索引失敗詳細:")
        print(_tb.format_exc())
        return None, f"❌ 建立向量索引失敗：{type(e).__name__}: {e}"

    if progress:
        progress(0.85, desc="💾 儲存 Embedding 至磁碟...")

    saved_ok = False
    try:
        cache.mkdir(parents=True, exist_ok=True)
        index.storage_context.persist(persist_dir=str(cache))
        if _cache_marker(cache).exists():
            saved_ok = True
    except Exception as e:
        print(f"⚠️ 快取儲存失敗：{e}")

    if progress:
        progress(0.95, desc="🔗 組裝查詢引擎...")

    engine = HierarchicalQueryEngine(index, nodes)

    if progress:
        progress(1.0, desc="✅ 完成！")

    status_msg = (
        f"✅ 索引建立完成（{len(nodes)} 個節點），已存入快取：{cache.name}"
        if saved_ok else
        f"⚠️ 索引建立完成（{len(nodes)} 個節點），但快取儲存失敗"
    )
    return engine, status_msg


# ==========================================
# 0-B. QA 答案快取管理
# ==========================================

QA_CACHE_FILE = "qa_cache.json"


def get_qa_cache_path(txt_path: str) -> Path:
    return Path(txt_path).parent / QA_CACHE_FILE


def load_qa_cache(txt_path: str) -> list:
    path = get_qa_cache_path(txt_path)
    if not path.exists():
        return []
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return []


def save_qa_entry(txt_path: str, question: str, answer: str,
                  related_clauses: list, complete_sections: list,
                  model: str, lang: str) -> str:
    """儲存一筆 QA，回傳 ID"""
    cache = load_qa_cache(txt_path)
    entry = {
        "id":                str(uuid.uuid4()),
        "question":          question,
        "answer":            answer,
        "related_clauses":   related_clauses,
        "complete_sections": complete_sections,
        "model":             model,
        "lang":              lang,
        "timestamp":         datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
    cache.append(entry)
    try:
        with open(get_qa_cache_path(txt_path), "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"⚠️ 儲存 QA 快取失敗：{e}")
    return entry["id"]


def delete_qa_entry_by_id(txt_path: str, entry_id: str) -> str:
    cache = load_qa_cache(txt_path)
    new_cache = [e for e in cache if e["id"] != entry_id]
    if len(new_cache) == len(cache):
        return f"⚠️ 找不到此 ID"
    try:
        with open(get_qa_cache_path(txt_path), "w", encoding="utf-8") as f:
            json.dump(new_cache, f, ensure_ascii=False, indent=2)
        return f"✅ 已刪除（ID: {entry_id[:12]}...）"
    except Exception as e:
        return f"❌ 刪除失敗：{e}"


def delete_folder_qa_cache(folder_name: str) -> str:
    folder = Path("./text_result") / folder_name
    qa_file = folder / QA_CACHE_FILE
    if qa_file.exists():
        qa_file.unlink()
        return f"✅ 已清除「{folder_name}」的所有問答快取"
    return f"⚠️「{folder_name}」沒有問答快取"


def delete_all_qa_cache() -> str:
    count = 0
    root = Path("./text_result")
    if root.exists():
        for folder in root.iterdir():
            qa_file = folder / QA_CACHE_FILE
            if qa_file.exists():
                qa_file.unlink()
                count += 1
    return f"✅ 已清除所有問答快取（共 {count} 個資料夾）"


def retrieve_similar_qa(question: str, txt_path: str, top_k: int = 3) -> list:
    """
    語意搜尋快取中最相似的前 top_k 筆問答。
    回傳 [(entry, score), ...] 依相似度降冪排序，無閾值限制。
    若快取為空則回傳 []。
    """
    cache = load_qa_cache(txt_path)
    if not cache:
        return []

    scored = []

    if RAG_AVAILABLE:
        try:
            from llama_index.core import Settings
            q_emb = np.array(Settings.embed_model.get_text_embedding(question))
            for entry in cache:
                c_emb = np.array(Settings.embed_model.get_text_embedding(entry["question"]))
                norm = np.linalg.norm(q_emb) * np.linalg.norm(c_emb)
                score = float(np.dot(q_emb, c_emb) / norm) if norm > 0 else 0.0
                scored.append((entry, score))
            scored.sort(key=lambda x: x[1], reverse=True)
            return scored[:top_k]
        except Exception as e:
            print(f"⚠️ 向量相似度計算失敗，退化為關鍵字重疊排序：{e}")

    # Fallback: character-level overlap ratio
    def _overlap(a: str, b: str) -> float:
        a_set, b_set = set(a.strip()), set(b.strip())
        if not a_set or not b_set:
            return 0.0
        return len(a_set & b_set) / max(len(a_set), len(b_set))

    for entry in cache:
        scored.append((entry, _overlap(question, entry["question"])))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


def fetch_qa_candidates(message: str, txt_path: str):
    """
    送出問題後先只做 embedding 檢索，回傳 top-3 快取問答候選。
    完全不呼叫 LLM。

    Returns:
        candidates       : list of (entry, score)
        group_visible    : gr.update for the candidates panel
        radio_choices    : list[str] for Radio widget
        radio_visible    : gr.update for Radio (hidden when empty)
        use_cache_visible: gr.update for "使用快取" btn (hidden when empty)
        candidates_html  : preview HTML string
    """
    if not txt_path:
        html = ("<div style='background:#fff3cd;padding:12px;border-radius:6px;'>"
                "⚠️ 尚未載入文件，無法檢索快取</div>")
        return ([], gr.update(visible=True),
                gr.update(visible=False), gr.update(visible=False), html)

    candidates = retrieve_similar_qa(message, txt_path, top_k=3)

    if not candidates:
        html = ("<div style='background:#f8f9fa;border:1px solid #dee2e6;"
                "padding:12px;border-radius:6px;'>"
                "📭 快取中找不到相似問答，請點擊「🔄 重新呼叫 AI 生成」</div>")
        return ([], gr.update(visible=True),
                gr.update(visible=False), gr.update(visible=False), html)

    # Build HTML preview cards
    html = (f"<div style='padding:6px;'>"
            f"<strong>🔍 找到 {len(candidates)} 筆相似歷史問答</strong>"
            f"<p style='font-size:0.82em;color:#888;margin:4px 0 6px;'>"
            f"從下方選取後點「✅ 使用此快取答案」，或點「🔄 重新呼叫 AI 生成」</p>")
    for i, (entry, score) in enumerate(candidates, 1):
        q  = entry.get("question", "")
        a  = entry.get("answer",   "")
        ts = entry.get("timestamp", "")
        n_c = len(entry.get("related_clauses",   []))
        n_s = len(entry.get("complete_sections", []))
        html += f"""
        <div style='border:1px solid #ced4da;margin:6px 0;padding:10px 12px;
                    border-radius:6px;background:#fff;'>
            <div style='display:flex;justify-content:space-between;margin-bottom:4px;'>
                <strong>#{i}</strong>
                <span style='font-size:0.82em;color:#6c757d;'>
                    相似度 {score:.3f} &nbsp;·&nbsp; {ts[:16]}
                    &nbsp;·&nbsp; {n_c} 條款 / {n_s} 章節
                </span>
            </div>
            <div style='background:#e8f4f8;padding:6px 8px;border-radius:4px;
                        font-size:0.88em;margin-bottom:4px;'>
                <strong>❓</strong> {q[:120]}{"..." if len(q)>120 else ""}
            </div>
            <div style='background:#f8f9fa;padding:6px 8px;border-radius:4px;
                        font-size:0.85em;color:#555;max-height:60px;overflow:hidden;'>
                {a[:180]}{"..." if len(a)>180 else ""}
            </div>
        </div>"""
    html += "</div>"

    choices = [
        f"#{i} 相似度 {score:.3f} — {entry['question'][:70]}{'...' if len(entry['question'])>70 else ''}"
        for i, (entry, score) in enumerate(candidates, 1)
    ]

    return (candidates, gr.update(visible=True),
            gr.update(visible=True, choices=choices, value=choices[0]),
            gr.update(visible=True), html)


def apply_cached_answer(candidates, selected_label, question,
                        history, use_complete_sections):
    """
    使用者選了某筆快取答案 → 更新 chatbot + 詳情面板（與 RAG 新答案完全一致的格式）。

    Returns: history, info_html, panel_update, pending_qa, cand_group_hidden
    """
    if not candidates:
        return (history,
                "<p style='color:red;'>⚠️ 沒有候選答案</p>",
                gr.update(visible=False), None, gr.update(visible=False))

    # Parse selected index
    idx = 0
    if selected_label:
        try:
            idx = int(selected_label.split()[0].replace("#", "")) - 1
        except Exception:
            idx = 0
    idx = max(0, min(idx, len(candidates) - 1))

    entry, score    = candidates[idx]
    response        = entry.get("answer",            "")
    rel_clauses     = entry.get("related_clauses",   [])
    complete_secs   = entry.get("complete_sections", [])
    orig_question   = entry.get("question",          "")

    cache_badge = (
        f"<div style='background:#d4edda;border:1px solid #28a745;"
        f"padding:10px;border-radius:4px;margin-bottom:12px;'>"
        f"<strong>💾 快取答案</strong>（相似度：{score:.3f}）"
        f"&nbsp;·&nbsp; 原問題：「{orig_question[:60]}{'...' if len(orig_question)>60 else ''}」"
        f"</div>"
    )
    rel_html = format_related_clauses(rel_clauses)
    sec_html = format_complete_sections(complete_secs) if use_complete_sections else \
        "<p style='color:#6c757d;'>（已關閉完整章節檢索）</p>"

    full_info_html = f"""
    <div style='padding:15px;'>
        {cache_badge}
        <div style='background:#d1ecf1;padding:12px;margin-bottom:15px;border-radius:4px;'>
            <strong>📊 快取資料：</strong> {len(rel_clauses)} 個條款
            {f", {len(complete_secs)} 個章節" if use_complete_sections else ""}
        </div>
        {rel_html}<hr>{sec_html}
    </div>"""

    history.append({"role": "user",
                    "content": question or orig_question})
    history.append({"role": "assistant",
                    "content": f"💾 *（快取答案，相似度 {score:.3f}）*\n\n{response}"})

    pending_qa = {
        "question":          question or orig_question,
        "answer":            response,
        "related_clauses":   rel_clauses,
        "complete_sections": complete_secs,
        "model":             entry.get("model", ""),
        "lang":              entry.get("lang",  ""),
    }

    return (history, full_info_html,
            gr.update(visible=True), pending_qa, gr.update(visible=False))


def render_qa_html(folder_name: str) -> str:
    """產生指定資料夾的 QA 快取 HTML 顯示"""
    if not folder_name:
        return "<p style='color:gray;padding:20px;'>請先選擇文件資料夾</p>"
    folder = Path("./text_result") / folder_name
    qa_file = folder / QA_CACHE_FILE
    if not qa_file.exists():
        return f"<p style='color:#888;padding:20px;'>「{folder_name}」尚無儲存的問答</p>"
    try:
        with open(qa_file, "r", encoding="utf-8") as f:
            entries = json.load(f)
    except Exception as e:
        return f"<p style='color:red;'>讀取失敗：{e}</p>"
    if not entries:
        return "<p style='color:gray;padding:20px;'>此資料夾沒有儲存的問答</p>"

    html = (
        f"<div style='padding:10px;'>"
        f"<h3>📚 {folder_name} — 共 {len(entries)} 筆問答</h3>"
        f"<p style='font-size:0.85em;color:#888;'>複製 ID 後可在下方輸入框刪除單筆</p>"
    )
    for i, e in enumerate(reversed(entries), 1):
        eid       = e.get("id", "")
        ts        = e.get("timestamp", "")
        q         = e.get("question", "")
        a         = e.get("answer", "")
        model     = e.get("model", "")
        n_clauses = len(e.get("related_clauses", []))
        n_secs    = len(e.get("complete_sections", []))
        html += f"""
        <div style='border:1px solid #dee2e6;margin:12px 0;padding:15px;border-radius:8px;background:#fff;'>
            <div style='display:flex;justify-content:space-between;flex-wrap:wrap;gap:4px;margin-bottom:8px;'>
                <span style='font-size:0.82em;color:#6c757d;'>#{len(entries)-i+1} &nbsp;·&nbsp; {ts} &nbsp;·&nbsp; 模型：{model}</span>
                <span style='font-size:0.78em;font-family:monospace;color:#aaa;cursor:pointer;'
                      title='點擊複製 ID' onclick="navigator.clipboard.writeText('{eid}')">
                    ID: {eid[:20]}... 📋
                </span>
            </div>
            <div style='background:#e8f4f8;padding:10px;border-radius:4px;margin-bottom:8px;'>
                <strong>❓ 問：</strong>{q}
            </div>
            <div style='background:#f8f9fa;padding:10px;border-radius:4px;max-height:110px;overflow-y:auto;'>
                <strong>💬 答：</strong>{a[:350]}{"..." if len(a) > 350 else ""}
            </div>
            <div style='margin-top:6px;font-size:0.82em;color:#555;'>
                📎 {n_clauses} 個條款 &nbsp;·&nbsp; {n_secs} 個章節
            </div>
        </div>"""
    html += "</div>"
    return html


# ==========================================
# 0-C. 文件圖片工具
# ==========================================

def get_doc_images(txt_path: str) -> list:
    """取得文件資料夾內按頁碼排序的所有頁面圖片"""
    if not txt_path or not os.path.exists(txt_path):
        return []
    folder = Path(txt_path).parent
    imgs = [p for p in (list(folder.glob("*.jpg")) + list(folder.glob("*.png")))
            if p.stem.startswith("page")]
    imgs.sort(
        key=lambda x: int(x.stem.replace("page", "")) if x.stem.replace("page", "").isdigit() else 999
    )
    return [str(p) for p in imgs]


def toggle_doc_images(txt_path: str, visible: bool):
    """切換文件圖片面板，載入或清空圖片"""
    new_visible = not visible
    if new_visible:
        images = get_doc_images(txt_path) if txt_path else []
        return gr.update(visible=True), gr.update(value=images), True
    return gr.update(visible=False), gr.update(value=[]), False


# ==========================================
# 0-D. 模型載入 (OCR + RAG)
# ==========================================

from transformers import AutoTokenizer, AutoProcessor, AutoModelForImageTextToText


def get_model_list():
    try:
        models_info = ollama.list()
        return [i["model"] for i in models_info["models"]]
    except Exception as e:
        print(f"Error fetching models: {e}")
        return ["error"]


# --- OCR 模型 ---
ocr_model_path = "./ocrmodel"
try:
    print("正在載入 OCR 模型...")
    ocr_model = AutoModelForImageTextToText.from_pretrained(
        ocr_model_path, torch_dtype="auto", device_map="auto"
    )
    ocr_model.eval()
    ocr_tokenizer = AutoTokenizer.from_pretrained(ocr_model_path)
    ocr_processor = AutoProcessor.from_pretrained(ocr_model_path)
    print("✅ OCR 模型載入完成")
except Exception as e:
    print(f"⚠️ OCR 模型載入失敗: {e}")
    ocr_model, ocr_tokenizer, ocr_processor = None, None, None

# --- RAG 模型 (LlamaIndex) ---
try:
    from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
    from llama_index.llms.ollama import Ollama
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
    from parsetooltestspeedfix import HierarchicalContractNodeParser, HierarchicalQueryEngine

    print("正在載入 RAG 模型 (LLM + Embedding)...")
    llm = Ollama(model="qwen3:4b", request_timeout=1200.0)
    embed_model = HuggingFaceEmbedding(
        model_name="IEITYuan/Yuan-embedding-2.0-zh", device="cuda"
    )
    Settings.llm = llm
    Settings.embed_model = embed_model

    RAG_AVAILABLE = True
    print("✅ RAG 模型載入完成")
except ImportError:
    print("❌ 缺少 llama_index 或 parsetool，Step 4 將無法使用。")
    RAG_AVAILABLE = False
except Exception as e:
    print(f"❌ RAG 模型初始化失敗: {e}")
    RAG_AVAILABLE = False


# ==========================================
# 智能文件檢索相關函數
# ==========================================

def build_folder_metadata_index():
    root = Path("./text_result")
    if not root.exists():
        return None, []
    documents, folder_info = [], []
    for folder in root.iterdir():
        if not folder.is_dir():
            continue
        folder_name = folder.name
        txt_files = list(folder.glob("*.txt"))
        if not txt_files:
            continue
        txt_path = txt_files[0]
        metadata = {
            "folder_name": folder_name,
            "txt_path": str(txt_path),
            "file_count": len(list(folder.glob("*.jpg"))) + len(list(folder.glob("*.png"))),
        }
        documents.append(Document(text=f"{folder_name}", metadata=metadata))
        folder_info.append(metadata)
    if not documents:
        return None, []
    index = VectorStoreIndex.from_documents(documents)
    return index, folder_info


def search_folders_by_description(description, index, top_k=5):
    if index is None:
        return []
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(description)
    return [
        {
            "folder_name": n.node.metadata.get("folder_name"),
            "txt_path":    n.node.metadata.get("txt_path"),
            "score":       n.score,
        }
        for n in nodes
    ]


def smart_search_and_confirm(
    doc_description, user_question, model, lang, use_context, folder_index,
    progress=gr.Progress()
):
    empty_returns = (
        None, gr.update(visible=False), gr.update(visible=False),
        gr.update(visible=False), None, [], None, gr.update(visible=False),
    )

    if not doc_description or not doc_description.strip():
        return (None, "<div style='color:orange;padding:20px;'><h3>⚠️ 請輸入檔案描述</h3></div>",
                *empty_returns[1:], "請先輸入檔案描述")

    if not user_question or not user_question.strip():
        return (None, "<div style='color:orange;padding:20px;'><h3>⚠️ 請輸入您的問題</h3></div>",
                *empty_returns[1:], "請先輸入問題")

    if folder_index is None:
        return (None, "<div style='color:red;padding:20px;'><h3>❌ 系統錯誤：文件索引未建立</h3></div>",
                *empty_returns[1:], "索引未建立")

    progress(0.3, desc="🔍 正在檢索相關文件...")
    results = search_folders_by_description(doc_description, folder_index, top_k=5)

    if not results:
        return (
            None,
            f"<div style='background:#fff3cd;border:2px solid #ffc107;padding:20px;border-radius:8px;'>"
            f"<h3>😔 未找到相關文件</h3><p>描述：{doc_description}</p></div>",
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            None, "未找到文件", [], None, gr.update(visible=False),
        )

    top = results[0]
    confirm_html = f"""
    <div style='background:#a7c4db;border:2px solid #0c5460;padding:20px;border-radius:8px;'>
        <h3>🎯 找到最相似的文件</h3>
        <div style='padding:15px;margin:10px 0;border-radius:5px;'>
            <h4>📄 {top['folder_name']}</h4>
            <p><strong>相似度分數：</strong>{top['score']:.3f}</p>
            <p><strong>檔案路徑：</strong>{top['txt_path']}</p>
        </div>
        <p style='font-size:1.1em;margin-top:15px;'><strong>❓ 這是您要找的文件嗎？</strong></p>
    </div>
    """
    progress(1.0, desc="等待使用者確認...")

    return (
        None, confirm_html,
        gr.update(visible=True), gr.update(visible=True), gr.update(visible=False),
        None, f"找到文件：{top['folder_name']}，等待確認...",
        results, None, gr.update(visible=False),
    )


def confirm_yes_and_load_doc(
    search_results, user_question, use_context, progress=gr.Progress()
):
    """
    使用者確認文件後：只載入 RAG 引擎並做快取候選檢索，不呼叫 LLM。

    Returns (15 values):
      confirm_yes_btn, confirm_no_btn, smart_qa_area,
      smart_rag_engine, smart_status, smart_retrieval_info, smart_info_panel,
      smart_txt_path_state,
      cand_group_visible, cand_html_value, cand_radio_choices,
      cand_radio_update, cand_use_btn_update,
      smart_qa_candidates, smart_pending_question
    """
    def _err(msg):
        return (
            gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=True), None, f"❌ {msg}", "", gr.update(visible=False),
            None,
            gr.update(visible=False), "", [], gr.update(visible=False),
            gr.update(visible=False), [], user_question,
        )

    if not search_results:
        return _err("沒有選中的文件")

    top         = search_results[0]
    txt_path    = top["txt_path"]
    folder_name = top["folder_name"]
    progress(0.2, desc=f"📂 正在載入文件：{folder_name}...")

    fake_state = {"original_file": "Smart Search", "txt_path": txt_path}

    try:
        engine, status = build_rag_index(fake_state, progress=progress)
        if engine is None:
            return _err(status)

        progress(0.85, desc="🔍 檢索快取問答...")
        candidates, cand_grp, radio_upd, use_btn_upd, cand_html = \
            fetch_qa_candidates(user_question, txt_path)

        progress(1.0, desc="完成！")
        return (
            gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=True), engine,
            f"✅ 已載入：{folder_name}", "", gr.update(visible=False),
            txt_path,
            cand_grp, cand_html, radio_upd, use_btn_upd,
            candidates, user_question,
        )
    except Exception as e:
        import traceback; traceback.print_exc()
        return _err(str(e))


def confirm_no_show_alternatives(search_results):
    if not search_results or len(search_results) < 2:
        return (
            "<div style='background:#f8d7da;border:2px solid #721c24;padding:20px;border-radius:8px;'>"
            "<h3>😔 沒有其他候選文件</h3></div>",
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
        )
    html = "<div style='padding:15px;'><h3>📋 其他候選文件</h3>"
    for i, result in enumerate(search_results[1:], 2):
        html += f"""
        <div style='border:1px solid #007bff;margin:10px 0;padding:15px;border-radius:5px;'>
            <h4>候選 {i}: {result['folder_name']}</h4>
            <p><strong>相似度分數：</strong>{result['score']:.3f}</p>
            <p><strong>路徑：</strong>{result['txt_path']}</p>
        </div>"""
    html += "</div>"
    return (html, gr.update(visible=True), gr.update(visible=False), gr.update(visible=False))


def select_alternative_and_load_doc(
    search_results, selected_idx, user_question, use_context,
    progress=gr.Progress()
):
    try:
        idx = int(selected_idx.split()[1]) - 1
    except Exception:
        idx = 1
    if idx >= len(search_results):
        idx = len(search_results) - 1
    return confirm_yes_and_load_doc(
        [search_results[idx]], user_question, use_context, progress
    )



# ==========================================
# 格式化輸出
# ==========================================

def format_related_clauses(clauses):
    if not clauses:
        return "<p>沒有找到相關條款</p>"
    html = f"<h3>🎯 相關條款 ({len(clauses)} 個)</h3>"
    for i, c in enumerate(clauses, 1):
        html += (
            f"<div style='border:1px solid #ddd;margin:10px 0;padding:10px;border-radius:5px;'>"
            f"<h4>{i}. 【{c.get('hierarchy_path','N/A')}】</h4>"
            f"<p><strong>標題:</strong> {c.get('clause_title','N/A')}</p>"
            f"<p><strong>類型:</strong> {c.get('clause_type','N/A')} | "
            f"<strong>相似度:</strong> {c.get('score',0.0):.3f}</p>"
            f"<p><strong>預覽:</strong> {c.get('text_preview','N/A')}</p>"
            f"</div>"
        )
    return html


def format_complete_sections(sections):
    if not sections:
        return "<p>沒有完整章節內容</p>"
    html = f"<h3>📋 完整章節 ({len(sections)} 個)</h3>"
    for s in sections:
        html += (
            f"<div style='border:2px solid #007bff;margin:15px 0;padding:15px;border-radius:8px;'>"
            f"<h4>章節 {s.get('section_number','')}: {s.get('section_title','')}</h4>"
            f"<p><strong>階層路徑:</strong> {s.get('hierarchy_path','')}</p>"
            f"<div style='background:grey;padding:10px;border-radius:4px;max-height:300px;overflow-y:auto;'>"
            f"<div style='white-space:pre-wrap;font-family:inherit;color:#333;line-height:1.5;'>"
            f"{s.get('full_content','')}</div></div></div>"
        )
    return html


# ==========================================
# OCR Prompt & 處理
# ==========================================

EXTRACT_PROMPT = (
    "Below is the image of one page of a document. "
    "Just return the plain text representation of this document as if you were reading it naturally.\n"
    "ALL tables should be presented in HTML format.\n"
    'If there are images or figures in the page, present them as "", '
    "(left,top,right,bottom) are the coordinates of the top-left and bottom-right corners.\n"
    "Present all titles and headings as H1 headings.\n"
    "Do not hallucinate.\n"
)


def ocr_page(image_path, model, processor, max_new_tokens=4096):
    image = Image.open(image_path)
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{image_path}"},
                {"type": "text",  "text":  EXTRACT_PROMPT},
            ],
        },
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt")
    inputs = inputs.to(model.device)
    output_ids = model.generate(**inputs, temperature=0.0, max_new_tokens=max_new_tokens, do_sample=False)
    generated_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
    output_text = processor.batch_decode(
        generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )
    return output_text[0]


# ==========================================
# 工具函數
# ==========================================

TEMP_DIR = Path("./temp_split_docs")
if TEMP_DIR.exists():
    shutil.rmtree(TEMP_DIR)
TEMP_DIR.mkdir(exist_ok=True)


def cleanup_temp_dir():
    if TEMP_DIR.exists():
        for item in TEMP_DIR.iterdir():
            if item.is_file():
                item.unlink()
            elif item.is_dir():
                shutil.rmtree(item)


def get_pdf_pages(pdf_path):
    doc = fitz.open(pdf_path)
    count = doc.page_count
    doc.close()
    return count


def generate_pdf_preview(pdf_path):
    try:
        pages = get_pdf_pages(pdf_path)
        return convert_from_path(pdf_path, dpi=120, first_page=1, last_page=pages)
    except Exception as e:
        print(f"Preview Error: {e}")
        return []


def docx_to_pdf_libreoffice(docx_path, output_dir):
    pdf_name = Path(docx_path).stem + ".pdf"
    output_path = output_dir / pdf_name
    try:
        subprocess.run(
            ["soffice", "--headless", "--convert-to", "pdf", "--outdir",
             str(output_dir), str(docx_path)],
            check=True, timeout=60,
        )
        if output_path.exists():
            return output_path
        raise FileNotFoundError("Conversion failed")
    except Exception as e:
        raise RuntimeError(f"DOCX to PDF Error: {e}")


# ==========================================
# 核心邏輯 (Process, Split, OCR)
# ==========================================

def process_upload(file, state_data):
    cleanup_temp_dir()
    fail = [
        None, None, None, "請上傳檔案", 1, 1,
        gr.update(visible=True), gr.update(visible=False),
        gr.update(visible=False), gr.update(visible=False), "",
    ]

    if file is None:
        return [gr.update(value="請先選擇檔案"), *fail[1:]]

    file_path = Path(file)
    file_stem = file_path.stem
    ext = file_path.suffix.lower()
    new_state = {
        "original_file": str(file_path), "file_ext": ext,
        "converted_pdf": None, "split_pdf_path": None, "txt_path": None,
    }

    try:
        if ext == ".pdf":
            pdf_path = str(file_path)
        elif ext == ".docx":
            pdf_path = str(docx_to_pdf_libreoffice(str(file_path), TEMP_DIR))
            new_state["converted_pdf"] = pdf_path
        else:
            return ["❌ 格式錯誤", *fail[1:]]

        cnt  = get_pdf_pages(pdf_path)
        prev = generate_pdf_preview(pdf_path)

        found_records = []
        result_dir = Path("./text_result")
        if result_dir.exists():
            for folder in result_dir.iterdir():
                if folder.is_dir() and file_stem in folder.name:
                    parts = folder.name.split("_")
                    range_info = parts[-1] if len(parts) > 1 else "未知範圍"
                    found_records.append(range_info)

        if found_records:
            history_msg = f"⚠️ 偵測到歷史處理紀錄 ({', '.join(found_records)})。推薦改名或確認範圍。"
            return [
                history_msg, new_state, prev, f"共 {cnt} 頁", 1, cnt,
                gr.update(visible=True), gr.update(visible=False),
                gr.update(visible=False), gr.update(visible=False), "",
            ]
        else:
            return [
                f"✅ 讀取成功 (共 {cnt} 頁)", new_state, prev,
                f"文件共 {cnt} 頁，請選擇切割範圍：", 1, cnt,
                gr.update(visible=False), gr.update(visible=True),
                gr.update(visible=False), gr.update(visible=False), "",
            ]

    except Exception as e:
        return [f"❌ 錯誤: {e}", *fail[1:]]


def split_document(state_data, start_page, end_page):
    fail = [
        "", None, state_data,
        gr.update(visible=True), gr.update(visible=False),
        gr.update(visible=False), gr.update(visible=False), "", None,
        gr.update(visible=False),
    ]

    if not state_data or not state_data.get("original_file"):
        return "❌ 請重傳", *fail[1:]

    state_data["txt_path"] = None
    target = (
        state_data["converted_pdf"]
        if state_data["file_ext"] == ".docx"
        else state_data["original_file"]
    )

    try:
        start = max(1, int(start_page))
        end   = int(end_page)

        doc = fitz.open(target)
        total_pages = doc.page_count
        if end > total_pages:
            end = total_pages
        if start > end:
            return f"❌ 範圍錯誤：起始頁 ({start}) 不能大於結束頁 ({end})", *fail[1:]

        out_name = f"split_{Path(state_data['original_file']).stem}_{start}-{end}.pdf"
        out_path = TEMP_DIR / out_name

        new_doc = fitz.open()
        for i in range(start - 1, end):
            new_doc.insert_pdf(doc, from_page=i, to_page=i)
        new_doc.save(str(out_path))
        doc.close(); new_doc.close()

        state_data["split_pdf_path"] = str(out_path)

        return [
            f"✅ 切割完成: {out_name}", str(out_path), state_data,
            gr.update(visible=False), gr.update(visible=True), gr.update(visible=True),
            "", None, gr.update(visible=False),
        ]
    except Exception as e:
        return f"❌ 失敗: {e}", *fail[1:]


def pdf_word_count(path):
    with pdfplumber.open(path) as pdf:
        if len(pdf.pages) > 0:
            text = pdf.pages[0].extract_text()
            return len(text) if text else 0
    return 0


def ocr_or_not(num_of_range, path):
    return num_of_range < pdf_word_count(path)


def pdf_process_and_save(pdf_path, progress=None):
    if progress:
        progress(0.1, desc="初始化...")
    fname = os.path.basename(pdf_path).split(".")[0]
    parts = fname.split("_")
    if len(parts) > 2:
        fname = "_".join(parts[1:-1])

    ocrpath = "./text_result/" + fname
    if os.path.exists(ocrpath):
        shutil.rmtree(ocrpath)
    os.makedirs(ocrpath, exist_ok=True)

    if progress:
        progress(0.2, desc="PDF 轉圖片...")
    pages = convert_from_path(pdf_path)
    pagelist = []
    for i, p in enumerate(pages):
        p_path = f"{ocrpath}/page{i}.jpg"
        pagelist.append(p_path)
        p.save(p_path, "JPEG")

    full_text = ""
    if progress:
        progress(0.4, desc="分析文字量...")

    if ocr_or_not(50, pdf_path):
        method = "PDFPlumber"
        if progress:
            progress(0.5, desc="提取原生文字...")
        with pdfplumber.open(pdf_path) as pdf:
            for p in pdf.pages:
                t = p.extract_text()
                if t:
                    full_text += t + "\n"
    else:
        method = "OCR"
        if progress:
            progress(0.5, desc="執行 AI OCR...")
        if ocr_model is None:
            full_text = "❌ 模型未載入"
        else:
            for i, p_path in enumerate(pagelist):
                if progress:
                    progress(0.5 + 0.4 * (i / len(pagelist)), desc=f"辨識第 {i+1} 頁...")
                try:
                    res = ocr_page(p_path, ocr_model, ocr_processor, 15000)
                    try:
                        j = json.loads(res)
                        full_text += j.get("natural_text", str(j))
                    except Exception:
                        full_text += res
                    full_text += "\n"
                except Exception:
                    full_text += f"\n[Error Page {i}]\n"

    txt_path = f"{ocrpath}/{fname}.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(full_text)

    if progress:
        progress(1.0, desc="完成")
    return f"完成 ({method})", txt_path


def run_ocr_extraction(state_data, progress=gr.Progress()):
    split_path = state_data.get("split_pdf_path")
    if not split_path:
        return "❌ 無切割檔", None, state_data, gr.update(visible=True), gr.update(visible=False)
    try:
        msg, txt_path = pdf_process_and_save(split_path, progress=progress)
        state_data["txt_path"] = txt_path
        return [
            f"✅ {msg}", txt_path, state_data,
            gr.update(visible=False), gr.update(visible=True),
        ]
    except Exception as e:
        return f"❌ 失敗: {e}", None, state_data, gr.update(visible=True), gr.update(visible=False)


# ==========================================
# RAG 核心函數 (Step 4)
# ==========================================

def build_rag_index(state_data, progress=None):
    txt_path = state_data.get("txt_path")
    if not txt_path or not os.path.exists(txt_path):
        return None, "❌ 找不到文字檔案，請先回上一步執行 OCR。"
    return build_or_load_index(txt_path, progress=progress)


def rag_chat_response(
    message, history, rag_engine, selected_model, lang,
    use_complete_sections, current_txt_path=None
):
    """
    直接呼叫 RAG 產生新答案（使用者選擇「重新呼叫 AI 生成」時使用）。
    Returns: history, full_info_html, panel_update, pending_qa, cand_group_hidden
    """
    from llama_index.core import Settings
    from llama_index.llms.ollama import Ollama

    if rag_engine is None:
        history.append({"role": "user",     "content": message})
        history.append({"role": "assistant","content": "⚠️ 知識庫未就緒。"})
        return (history,
                "<div style='color:orange;padding:20px;'>⚠️ 知識庫未就緒</div>",
                gr.update(visible=False), None, gr.update(visible=False))

    try:
        combined_prompt = f"{message}\n\n(請注意：請務必使用「{lang}」回答我。)"
        Settings.llm = Ollama(model=selected_model, request_timeout=1200.0)

        rag_result = rag_engine.query_with_complete_sections(
            combined_prompt, include_complete_sections=use_complete_sections
        )

        response      = str(rag_result["answer"])
        rel_clauses   = rag_result.get("related_clauses",   [])
        complete_secs = rag_result.get("complete_sections", [])

        rel_html = format_related_clauses(rel_clauses)
        sec_html = format_complete_sections(complete_secs) if use_complete_sections else \
            "<p style='color:#6c757d;'>（已關閉完整章節檢索）</p>"

        full_info_html = f"""
        <div style='padding:15px;'>
            <div style='background:#d1ecf1;padding:12px;margin-bottom:15px;border-radius:4px;'>
                <strong>📊 找到：</strong> {len(rel_clauses)} 個條款
                {f", {len(complete_secs)} 個章節" if use_complete_sections else ""}
            </div>
            {rel_html}<hr>{sec_html}
        </div>"""

        pending_qa = {
            "question":          message,
            "answer":            response,
            "related_clauses":   rel_clauses,
            "complete_sections": complete_secs,
            "model":             selected_model,
            "lang":              lang,
        }

    except Exception as e:
        import traceback; traceback.print_exc()
        response       = f"❌ 錯誤: {str(e)}"
        full_info_html = f"<div style='color:red;padding:20px;'>錯誤：{str(e)}</div>"
        pending_qa     = None

    history.append({"role": "user",     "content": message})
    history.append({"role": "assistant","content": response})
    return (history, full_info_html,
            gr.update(visible=True), pending_qa, gr.update(visible=False))


def save_current_qa(pending_qa, current_txt_path: str) -> str:
    """使用者按「💾 儲存此答案」後呼叫，將 pending_qa 寫入 qa_cache.json。"""
    if not pending_qa:
        return "⚠️ 沒有可儲存的答案（請先提問）"
    if not current_txt_path:
        return "⚠️ 找不到文件路徑，無法儲存"
    try:
        entry_id = save_qa_entry(
            current_txt_path,
            pending_qa["question"],
            pending_qa["answer"],
            pending_qa.get("related_clauses",   []),
            pending_qa.get("complete_sections", []),
            pending_qa.get("model", ""),
            pending_qa.get("lang",  ""),
        )
        return f"✅ 已儲存（ID: {entry_id[:12]}...）"
    except Exception as e:
        return f"❌ 儲存失敗：{e}"


# ==========================================
# 導航與重設
# ==========================================

def back_to_step1():
    return [gr.update(visible=True), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False)]


def back_to_step2():
    return [gr.update(visible=False), gr.update(visible=True),
            gr.update(visible=False), gr.update(visible=False)]


def back_to_step3():
    return [gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=True), gr.update(visible=False)]


def get_history_folders():
    root = Path("./text_result")
    if not root.exists():
        return []
    return [d.name for d in root.iterdir() if d.is_dir()]


def preview_history_folder(folder_name):
    if not folder_name:
        return [], "請選擇資料夾"
    folder_path = Path("./text_result") / folder_name
    images = []
    try:
        jpgs = list(folder_path.glob("*.jpg"))
        jpgs.sort(
            key=lambda x: int(x.stem.replace("page", "")) if x.stem.replace("page", "").isdigit() else x.stem
        )
        images = [str(p) for p in jpgs]
    except Exception:
        images = [str(p) for p in folder_path.glob("*.jpg")]

    txt_files = list(folder_path.glob("*.txt"))
    status_msg = (
        f"📂 資料夾：{folder_name}\n"
        f"📸 圖片數：{len(images)}\n"
        f"📝 文字檔：{'✅ 存在' if txt_files else '❌ 遺失'}"
    )
    return images, status_msg


def load_history_and_go_to_step4(folder_name, progress=gr.Progress()):
    if not folder_name:
        current_models = get_model_list()
        yield (
            gr.update(visible=True), gr.update(visible=False),
            gr.update(value=[]), "❌ 請先選擇一個資料夾",
            None, gr.update(choices=current_models), False,
            gr.update(visible=False), None,
        )
        return

    folder_path = Path("./text_result") / folder_name
    txt_files   = list(folder_path.glob("*.txt"))

    if not txt_files:
        current_models = get_model_list()
        yield (
            gr.update(visible=True), gr.update(visible=False),
            gr.update(value=[]), "❌ 找不到 .txt 文字檔",
            None, gr.update(choices=current_models), False,
            gr.update(visible=False), None,
        )
        return

    txt_path       = str(txt_files[0])
    fake_state     = {"original_file": "History Load", "txt_path": txt_path}
    current_models = get_model_list()
    default_model  = "qwen3:4b" if "qwen3:4b" in current_models else current_models[0]

    yield (
        gr.update(visible=False), gr.update(visible=True),
        gr.update(value=[]), f"⏳ 正在載入：{folder_name}...",
        None, gr.update(choices=current_models, value=default_model),
        False, gr.update(visible=False), None,
    )

    try:
        engine, status = build_rag_index(fake_state, progress=progress)
        yield (
            gr.update(visible=False), gr.update(visible=True),
            gr.update(value=[]), status, engine,
            gr.update(choices=current_models), False,
            gr.update(visible=False), txt_path,
        )
    except Exception as e:
        yield (
            gr.update(visible=False), gr.update(visible=True),
            gr.update(value=[]), f"❌ 載入失敗: {str(e)}",
            None, gr.update(choices=current_models), False,
            gr.update(visible=False), None,
        )


def go_to_step4(state_data, progress=gr.Progress()):
    txt_path = state_data.get("txt_path")
    if not txt_path:
        yield [gr.update(visible=True), gr.update(visible=False),
               gr.update(value=[]), "❌ 找不到文字檔", None, gr.update(), None]
        return

    current_models = get_model_list()
    default_model  = "qwen3:4b" if "qwen3:4b" in current_models else current_models[0]

    yield (
        gr.update(visible=False), gr.update(visible=True),
        gr.update(value=[]), "⏳ 正在自動建立知識庫...", None,
        gr.update(choices=current_models, value=default_model), None,
    )

    try:
        engine, status = build_rag_index(state_data, progress=progress)
        yield (
            gr.update(visible=False), gr.update(visible=True),
            gr.update(value=[]), status, engine,
            gr.update(choices=current_models), txt_path,
        )
    except Exception as e:
        yield (
            gr.update(visible=False), gr.update(visible=True),
            gr.update(value=[]), f"❌ 失敗: {str(e)}",
            None, gr.update(choices=current_models), None,
        )


def reset_all():
    cleanup_temp_dir()
    empty_state = {
        "original_file": None, "file_ext": None,
        "converted_pdf": None, "split_pdf_path": None, "txt_path": None,
    }
    return [
        gr.update(value=None, interactive=True), empty_state, "",
        1, 1,
        gr.update(visible=True), gr.update(visible=False),
        gr.update(visible=False), gr.update(visible=False),
        None, "", "", None, "", None,
        gr.update(visible=True), "尚未建立索引", None, [],
        "", gr.update(visible=False), "中文", False, True,
        None,                    # step4_txt_path_state
        False,                   # doc_img_visible_4
        gr.update(visible=False), gr.update(value=[]),  # doc_img_group_4, doc_img_gallery_4
        None,                    # step4_pending_qa
        "",                      # save_qa_msg_4
    ]


# ==========================================
# Gradio 介面
# ==========================================

with gr.Blocks(title="智能文件處理系統") as demo:
    gr.Markdown("# 📄 智能合約處理系統")

    file_state         = gr.State(value={"original_file": None, "file_ext": None,
                                         "converted_pdf": None, "split_pdf_path": None,
                                         "txt_path": None})
    rag_engine_state   = gr.State(value=None)
    rag_result_cache   = gr.State(value={})
    info_visible_state = gr.State(value=False)

    # ── Step 4 image panel states ──────────────────────────────────────────
    step4_txt_path_state = gr.State(value=None)
    doc_img_visible_4    = gr.State(value=False)
    step4_pending_qa     = gr.State(value=None)
    step4_qa_candidates  = gr.State(value=[])      # top-3 候選快取答案
    step4_pending_q      = gr.State(value="")      # 暫存目前提問

    # ── Smart Tab states ───────────────────────────────────────────────────
    smart_txt_path_state  = gr.State(value=None)
    smart_img_visible     = gr.State(value=False)
    smart_pending_qa      = gr.State(value=None)
    smart_qa_candidates   = gr.State(value=[])     # top-3 候選快取答案
    smart_pending_q       = gr.State(value="")     # 暫存目前提問

    with gr.Tabs():

        # ──────────────────────────────────────────────────────────────────
        # Tab 1: 智能檢索回答
        # ──────────────────────────────────────────────────────────────────
        with gr.Tab("🔍 智能檢索回答"):
            gr.Markdown("## 智能文件問答")

            folder_index_state   = gr.State(value=None)
            search_results_state = gr.State(value=[])
            smart_rag_engine     = gr.State(value=None)
            smart_info_visible   = gr.State(value=False)
            smart_user_question  = gr.State(value="")

            with gr.Row():
                smart_model_selector = gr.Dropdown(
                    choices=get_model_list(), value="qwen3:4b", label="🤖 AI 模型", scale=1
                )
                smart_lang_input = gr.Textbox(label="🌐 輸出語言", value="中文", scale=1)
                smart_status     = gr.Textbox(
                    label="📡 系統狀態", value="準備就緒", interactive=False, scale=2
                )
                refresh_index_btn = gr.Button("🔄 更新檢索範圍", variant="secondary", scale=1)

            with gr.Group():
                gr.Markdown("### 📝 請輸入檔案描述與您的問題")
                with gr.Row():
                    doc_description_input  = gr.Textbox(
                        label="📁 檔案描述", placeholder="例如：合約1、租賃合約...", scale=1
                    )
                    smart_use_context_chk  = gr.Checkbox(
                        label="包含完整章節(生成時間較長)", value=True, scale=0, min_width=150
                    )
                user_question_input = gr.Textbox(
                    label="❓ 您的問題", placeholder="請輸入關於此合約的問題...", lines=3
                )
                smart_search_btn = gr.Button("🚀 開始智能搜尋與問答", variant="primary", size="lg")

            with gr.Group() as confirm_group:
                confirm_display = gr.HTML(value="")
                with gr.Row(visible=False) as confirm_buttons:
                    confirm_yes_btn = gr.Button("✅ 是的，就是這個文件", variant="primary",   scale=1)
                    confirm_no_btn  = gr.Button("❌ 不是，顯示其他選項", variant="secondary", scale=1)
                with gr.Group(visible=False) as alternative_area:
                    alternative_display = gr.HTML(value="")
                    with gr.Row():
                        alternative_selector = gr.Radio(
                            choices=["候選 2", "候選 3", "候選 4", "候選 5"],
                            value="候選 2", label="選擇其他文件"
                        )
                        select_alternative_btn = gr.Button("✔️ 確認選擇", variant="primary")

            gr.Markdown("---")

            with gr.Group(visible=False) as smart_qa_area:
                gr.Markdown("## 💬 問答結果")

                # 圖片預覽（智能搜尋 Tab）
                with gr.Row():
                    smart_show_img_btn = gr.Button("📸 顯示/隱藏文件圖片", variant="secondary", scale=0)

                with gr.Group(visible=False) as smart_img_group:
                    smart_img_gallery = gr.Gallery(
                        label="文件原稿圖片", columns=4, height=350, object_fit="contain"
                    )

                # ── QA 候選面板（快取檢索結果）────────────────────────────
                with gr.Group(visible=False) as smart_cand_group:
                    smart_cand_html  = gr.HTML(value="")
                    smart_cand_radio = gr.Radio(choices=[], label="選擇快取答案",
                                                visible=False)
                    with gr.Row():
                        smart_use_cache_btn = gr.Button(
                            "✅ 使用此快取答案", variant="primary",   visible=False, scale=1)
                        smart_regen_btn     = gr.Button(
                            "🔄 重新呼叫 AI 生成", variant="secondary", scale=1)

                with gr.Row(equal_height=True):
                    with gr.Column(scale=3):
                        smart_chatbot = gr.Chatbot(
                            type="messages", label="智能助手", height=500
                        )
                        smart_followup_msg = gr.Textbox(
                            placeholder="繼續提問...", show_label=False, container=False
                        )
                        with gr.Row():
                            smart_info_toggle = gr.Button(
                                "👁️ 顯示/隱藏 檢索詳情", variant="secondary"
                            )
                            smart_send_btn = gr.Button("🚀 發送", variant="primary")
                        with gr.Row():
                            save_qa_btn_smart = gr.Button("💾 儲存此答案", variant="secondary", scale=1)
                            save_qa_msg_smart = gr.Textbox(label="", value="", interactive=False,
                                                           show_label=False, container=False, scale=3)
                    with gr.Column(scale=2, visible=False) as smart_info_panel:
                        gr.Markdown("### 🔍 檢索原始條款")
                        smart_retrieval_info = gr.HTML(
                            value="<p style='color:gray;'>等待查詢...</p>"
                        )

        # ──────────────────────────────────────────────────────────────────
        # Tab 2: 檔案上傳與手動搜尋
        # ──────────────────────────────────────────────────────────────────
        with gr.Tab("📁 檔案上傳與手動搜尋"):

            with gr.Group(visible=True) as step1:
                with gr.Tabs():
                    with gr.Tab("🆕 上傳新文件"):
                        gr.Markdown("## 1️⃣ 上傳檔案")
                        file_input    = gr.File(
                            label="PDF / DOCX", file_types=[".pdf", ".docx"], type="filepath"
                        )
                        upload_msg    = gr.Textbox(label="狀態", interactive=False, show_label=False)
                        next_step_btn = gr.Button("▶️ 下一步：轉換與預覽", variant="primary")

                    with gr.Tab("📂 選擇歷史紀錄"):
                        gr.Markdown("## 讀取已處理的 OCR 結果")
                        with gr.Row():
                            history_dropdown = gr.Dropdown(
                                label="選擇歷史專案",
                                choices=get_history_folders(), interactive=True
                            )
                            refresh_hist_btn = gr.Button("🔄 重新整理清單", size="sm")
                        history_msg     = gr.Textbox(label="資料夾資訊", lines=3, interactive=False)
                        gr.Markdown("### 圖片預覽")
                        history_gallery = gr.Gallery(
                            label="內容預覽", columns=6, height=200, object_fit="contain"
                        )
                        load_history_btn = gr.Button(
                            "🚀 載入此專案並開始問答 (跳至 Step 4)", variant="primary"
                        )

            with gr.Group(visible=False) as step2:
                gr.Markdown("## 2️⃣ 預覽與切割")
                preview_gallery = gr.Gallery(
                    label="預覽", columns=5, height=250, object_fit="contain"
                )
                page_hint = gr.Markdown("頁數資訊")
                with gr.Row():
                    page_start = gr.Number(label="起始頁數", value=1, precision=0, minimum=1, step=1)
                    page_end   = gr.Number(label="結束頁數", value=1, precision=0, minimum=1, step=1)
                with gr.Row():
                    back_btn_2_1 = gr.Button("◀️ 返回",     variant="secondary")
                    split_btn    = gr.Button("🔪 執行切割", variant="primary")

            with gr.Group(visible=False) as step3:
                gr.Markdown("## 3️⃣ 切割結果與 OCR")
                result_msg = gr.Textbox(label="系統訊息", interactive=False)
                with gr.Row():
                    result_file     = gr.File(label="切割後的 PDF")
                    ocr_result_file = gr.File(label="文字提取結果 (.txt)")
                with gr.Row():
                    back_btn_3_2 = gr.Button("◀️ 返回切割",           variant="secondary")
                    ocr_btn      = gr.Button("🔍 執行 OCR / 文字提取", variant="primary")
                    to_rag_btn   = gr.Button(
                        "▶️ 下一步：智能問答 (RAG)", variant="primary", visible=False
                    )

            with gr.Group(visible=False) as step4:
                gr.Markdown("## 4️⃣ 智能合約問答 (RAG)")
                with gr.Row():
                    rag_status     = gr.Textbox(
                        label="📡 知識庫狀態", value="尚未建立索引",
                        interactive=False, scale=2
                    )
                    model_selector = gr.Dropdown(
                        choices=get_model_list(), value="qwen3:4b",
                        label="🤖 選擇 AI 模型", scale=1
                    )
                    lang_input = gr.Textbox(
                        label="🌐 輸出語言", value="中文",
                        placeholder="例如：中文...", scale=1
                    )

                # 文件圖片預覽（Step 4）
                with gr.Row():
                    show_img_btn_4 = gr.Button(
                        "📸 顯示/隱藏文件圖片", variant="secondary", scale=0
                    )

                with gr.Group(visible=False) as doc_img_group_4:
                    doc_img_gallery_4 = gr.Gallery(
                        label="文件原稿圖片", columns=4, height=350, object_fit="contain"
                    )

                # ── QA 候選面板（快取檢索結果）────────────────────────────
                with gr.Group(visible=False) as step4_cand_group:
                    step4_cand_html  = gr.HTML(value="")
                    step4_cand_radio = gr.Radio(choices=[], label="選擇快取答案",
                                                visible=False)
                    with gr.Row():
                        step4_use_cache_btn = gr.Button(
                            "✅ 使用此快取答案", variant="primary",   visible=False, scale=1)
                        step4_regen_btn     = gr.Button(
                            "🔄 重新呼叫 AI 生成", variant="secondary", scale=1)

                with gr.Row(equal_height=True):
                    with gr.Column(scale=3):
                        chatbot = gr.Chatbot(
                            type="messages", label="智能合約助手",
                            height=550, show_label=False
                        )
                        msg = gr.Textbox(
                            label="問答輸入",
                            placeholder="請輸入關於此合約的問題...",
                            show_label=False, container=False
                        )
                        with gr.Row(variant="panel"):
                            use_full_context_chk = gr.Checkbox(
                                label="包含完整章節(生成時間較長)", value=True, interactive=True,
                                info="若勾選，將同時檢索相關條款所在的完整章節內容。"
                            )
                        with gr.Row():
                            back_btn_4_3    = gr.Button("◀️ 返回上一步",         variant="secondary", scale=1)
                            info_toggle_btn = gr.Button("👁️ 顯示/隱藏 檢索相關條款", variant="secondary", scale=1)
                            send_btn        = gr.Button("🚀 發送問題",             variant="primary",   scale=2)
                        with gr.Row():
                            save_qa_btn_4  = gr.Button("💾 儲存此答案",           variant="secondary", scale=1)
                            save_qa_msg_4  = gr.Textbox(label="", value="", interactive=False,
                                                        show_label=False, container=False, scale=3)

                    with gr.Column(scale=2, visible=False) as side_info_panel:
                        gr.Markdown("### 🔍 檢索原始條款")
                        retrieval_info_area = gr.HTML(
                            value="<div style='color:#a7c4db;padding:20px;'>"
                                  "詢問問題後，相關的合約原文將顯示於此...</div>"
                        )

            gr.Markdown("---")
            reset_btn = gr.Button("🔄 重新開始", variant="secondary")

        # ──────────────────────────────────────────────────────────────────
        # Tab 3: Embedding 快取管理
        # ──────────────────────────────────────────────────────────────────
        with gr.Tab("🗃️ Embedding 快取管理"):
            gr.Markdown("## 本地 Embedding 快取管理")
            gr.Markdown(
                "每個文件首次載入時會計算 Embedding 並存成本地 JSON 檔案（`./text_result/<文件>/`）。"
                "之後再次載入同一文件時，直接讀取快取，**大幅加速**，無需重新計算。"
            )
            with gr.Row():
                refresh_cache_btn    = gr.Button("🔄 查看所有快取",   variant="secondary")
                delete_all_cache_btn = gr.Button("🗑️ 刪除全部快取",  variant="stop")

            cache_list_display = gr.JSON(label="目前快取清單", value=[])
            cache_op_msg       = gr.Textbox(label="操作結果", interactive=False)

            with gr.Row():
                cache_folder_dropdown   = gr.Dropdown(
                    label="選擇要刪除快取的文件",
                    choices=get_history_folders(), interactive=True
                )
                delete_single_cache_btn = gr.Button("🗑️ 刪除此文件的快取", variant="secondary")

        # ──────────────────────────────────────────────────────────────────
        # Tab 4: QA 問答快取管理
        # ──────────────────────────────────────────────────────────────────
        with gr.Tab("💬 問答快取管理"):
            gr.Markdown("## 已儲存問答快取管理")
            gr.Markdown(
                "每次問答結束後系統會自動儲存問題與答案（含檢索條款）到對應文件的資料夾中，"
                "下次提問相似問題時會直接從快取讀取，**跳過 AI 生成**，大幅加速。"
            )

            with gr.Row():
                qa_refresh_btn    = gr.Button("🔄 重新整理",      variant="secondary")
                qa_delete_all_btn = gr.Button("🗑️ 刪除所有快取", variant="stop")
            qa_op_msg = gr.Textbox(label="操作結果", interactive=False)

            gr.Markdown("### 選擇文件資料夾查看 / 刪除")
            with gr.Row():
                qa_folder_dropdown   = gr.Dropdown(
                    label="選擇文件資料夾",
                    choices=get_history_folders(), interactive=True, scale=3
                )
                qa_delete_folder_btn = gr.Button("🗑️ 刪除此資料夾的快取", variant="secondary", scale=1)

            qa_entries_html = gr.HTML(
                value="<p style='color:gray;padding:20px;'>請先選擇文件資料夾</p>"
            )

            gr.Markdown("### 刪除單筆答案")
            gr.Markdown(
                "點擊上方 ID 旁的 📋 按鈕複製後，貼到下方輸入框刪除指定條目。"
            )
            with gr.Row():
                qa_entry_id_input   = gr.Textbox(
                    label="輸入要刪除的完整 ID（UUID）",
                    placeholder="xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx",
                    scale=3
                )
                qa_delete_entry_btn = gr.Button("🗑️ 刪除此筆答案", variant="secondary", scale=1)

    # ==========================================
    # Events 事件綁定
    # ==========================================

    # Step 1 → 2
    next_step_btn.click(
        process_upload,
        inputs=[file_input, file_state],
        outputs=[upload_msg, file_state, preview_gallery, page_hint,
                 page_start, page_end, step1, step2, step3, step4, result_msg],
    )

    # Step 2 → 3
    split_btn.click(
        split_document,
        inputs=[file_state, page_start, page_end],
        outputs=[result_msg, result_file, file_state, step2, step3,
                 ocr_btn, result_msg, ocr_result_file, to_rag_btn],
    )

    # OCR
    ocr_btn.click(
        run_ocr_extraction,
        [file_state],
        [result_msg, ocr_result_file, file_state, ocr_btn, to_rag_btn]
    )

    # Step 3 → Step 4（新增 step4_txt_path_state 輸出）
    to_rag_btn.click(
        go_to_step4,
        inputs=[file_state],
        outputs=[step3, step4, chatbot, rag_status, rag_engine_state,
                 model_selector, step4_txt_path_state],
    )

    # Step 4 圖片預覽切換
    show_img_btn_4.click(
        toggle_doc_images,
        inputs=[step4_txt_path_state, doc_img_visible_4],
        outputs=[doc_img_group_4, doc_img_gallery_4, doc_img_visible_4],
    )

    # Step 4 側邊資訊面板切換
    info_toggle_btn.click(
        lambda visible: (gr.update(visible=not visible), not visible),
        inputs=[info_visible_state],
        outputs=[side_info_panel, info_visible_state],
    )

    # 導航返回
    back_btn_2_1.click(back_to_step1, [], [step1, step2, step3, step4])
    back_btn_3_2.click(back_to_step2, [], [step1, step2, step3, step4])
    back_btn_4_3.click(back_to_step3, [], [step1, step2, step3, step4])

    # 重設（更新輸出清單以包含新 state）
    reset_btn.click(
        reset_all, inputs=[],
        outputs=[
            file_input, file_state, upload_msg, page_start, page_end,
            step1, step2, step3, step4, preview_gallery, page_hint,
            result_msg, result_file, result_msg, ocr_result_file, ocr_btn,
            rag_status, rag_engine_state, chatbot, retrieval_info_area,
            side_info_panel, lang_input, info_visible_state, use_full_context_chk,
            step4_txt_path_state, doc_img_visible_4, doc_img_group_4, doc_img_gallery_4,
            step4_pending_qa, save_qa_msg_4,
        ],
    )

    # Step 4 問答 ── 先 fetch 候選，不呼叫 LLM
    def _fetch_step4(message, txt_path):
        cands, grp, radio_upd, use_btn_upd, html = \
            fetch_qa_candidates(message, txt_path)
        return cands, grp, html, radio_upd, use_btn_upd, message

    def _fetch_smart(message, txt_path):
        cands, grp, radio_upd, use_btn_upd, html = \
            fetch_qa_candidates(message, txt_path)
        return cands, grp, html, radio_upd, use_btn_upd, message

    msg.submit(
        _fetch_step4,
        inputs=[msg, step4_txt_path_state],
        outputs=[step4_qa_candidates, step4_cand_group, step4_cand_html,
                 step4_cand_radio, step4_use_cache_btn, step4_pending_q],
    ).then(lambda: "", None, msg)

    send_btn.click(
        _fetch_step4,
        inputs=[msg, step4_txt_path_state],
        outputs=[step4_qa_candidates, step4_cand_group, step4_cand_html,
                 step4_cand_radio, step4_use_cache_btn, step4_pending_q],
    ).then(lambda: "", None, msg)

    # Step 4 ── 使用快取答案
    step4_use_cache_btn.click(
        apply_cached_answer,
        inputs=[step4_qa_candidates, step4_cand_radio, step4_pending_q,
                chatbot, use_full_context_chk],
        outputs=[chatbot, retrieval_info_area, side_info_panel,
                 step4_pending_qa, step4_cand_group],
    )

    # Step 4 ── 重新呼叫 AI 生成
    step4_regen_btn.click(
        rag_chat_response,
        inputs=[step4_pending_q, chatbot, rag_engine_state, model_selector,
                lang_input, use_full_context_chk, step4_txt_path_state],
        outputs=[chatbot, retrieval_info_area, side_info_panel,
                 step4_pending_qa, step4_cand_group],
    )

    save_qa_btn_4.click(
        save_current_qa,
        inputs=[step4_pending_qa, step4_txt_path_state],
        outputs=[save_qa_msg_4],
    )

    # 歷史記錄相關
    history_dropdown.change(
        preview_history_folder,
        inputs=[history_dropdown],
        outputs=[history_gallery, history_msg]
    )
    refresh_hist_btn.click(
        lambda: gr.update(choices=get_history_folders()),
        inputs=[], outputs=[history_dropdown]
    )

    load_history_btn.click(
        load_history_and_go_to_step4,
        inputs=[history_dropdown],
        outputs=[step1, step4, chatbot, rag_status, rag_engine_state,
                 model_selector, info_visible_state, side_info_panel,
                 step4_txt_path_state],
    )

    # ── 智能檢索 Tab 事件 ──────────────────────────────────────────────────

    demo.load(
        lambda: build_folder_metadata_index()[0],
        inputs=[], outputs=[folder_index_state]
    )

    refresh_index_btn.click(
        lambda: (build_folder_metadata_index()[0], "✅ 檢索範圍已更新！"),
        inputs=[], outputs=[folder_index_state, smart_status],
    )

    smart_search_btn.click(
        smart_search_and_confirm,
        inputs=[doc_description_input, user_question_input, smart_model_selector,
                smart_lang_input, smart_use_context_chk, folder_index_state],
        outputs=[smart_chatbot, confirm_display, confirm_yes_btn, confirm_no_btn,
                 smart_qa_area, smart_rag_engine, smart_status,
                 search_results_state, smart_retrieval_info, smart_info_panel],
    ).then(
        lambda q: q, inputs=[user_question_input], outputs=[smart_user_question]
    ).then(
        lambda: (gr.update(visible=True), gr.update(visible=False)),
        outputs=[confirm_buttons, alternative_area],
    )

    # 確認文件 → 載入引擎 + 候選檢索（不呼叫 LLM）
    # Returns 14: confirm_yes_btn, confirm_no_btn, smart_qa_area,
    #   smart_rag_engine, smart_status, smart_retrieval_info, smart_info_panel,
    #   smart_txt_path_state, cand_grp, cand_html, cand_radio, cand_use_btn,
    #   smart_qa_candidates, smart_pending_q
    _CONFIRM_OUTPUTS = [
        confirm_yes_btn, confirm_no_btn, smart_qa_area,
        smart_rag_engine, smart_status, smart_retrieval_info, smart_info_panel,
        smart_txt_path_state,
        smart_cand_group, smart_cand_html,
        smart_cand_radio, smart_use_cache_btn,
        smart_qa_candidates, smart_pending_q,
    ]

    confirm_yes_btn.click(
        confirm_yes_and_load_doc,
        inputs=[search_results_state, smart_user_question, smart_use_context_chk],
        outputs=_CONFIRM_OUTPUTS,
    ).then(
        lambda: gr.update(visible=True), outputs=[smart_qa_area],
    )

    confirm_no_btn.click(
        confirm_no_show_alternatives,
        inputs=[search_results_state],
        outputs=[alternative_display, alternative_area, confirm_yes_btn, confirm_no_btn],
    )

    select_alternative_btn.click(
        select_alternative_and_load_doc,
        inputs=[search_results_state, alternative_selector,
                smart_user_question, smart_use_context_chk],
        outputs=_CONFIRM_OUTPUTS,
    ).then(
        lambda: gr.update(visible=True), outputs=[smart_qa_area],
    )

    # 智能 Tab ── 後續追問：先 fetch 候選
    smart_followup_msg.submit(
        _fetch_smart,
        inputs=[smart_followup_msg, smart_txt_path_state],
        outputs=[smart_qa_candidates, smart_cand_group, smart_cand_html,
                 smart_cand_radio, smart_use_cache_btn, smart_pending_q],
    ).then(lambda: "", None, smart_followup_msg)

    smart_send_btn.click(
        _fetch_smart,
        inputs=[smart_followup_msg, smart_txt_path_state],
        outputs=[smart_qa_candidates, smart_cand_group, smart_cand_html,
                 smart_cand_radio, smart_use_cache_btn, smart_pending_q],
    ).then(lambda: "", None, smart_followup_msg)

    # 智能 Tab ── 使用快取答案
    smart_use_cache_btn.click(
        apply_cached_answer,
        inputs=[smart_qa_candidates, smart_cand_radio, smart_pending_q,
                smart_chatbot, smart_use_context_chk],
        outputs=[smart_chatbot, smart_retrieval_info, smart_info_panel,
                 smart_pending_qa, smart_cand_group],
    )

    # 智能 Tab ── 重新呼叫 AI 生成
    smart_regen_btn.click(
        rag_chat_response,
        inputs=[smart_pending_q, smart_chatbot, smart_rag_engine,
                smart_model_selector, smart_lang_input, smart_use_context_chk,
                smart_txt_path_state],
        outputs=[smart_chatbot, smart_retrieval_info, smart_info_panel,
                 smart_pending_qa, smart_cand_group],
    )

    save_qa_btn_smart.click(
        save_current_qa,
        inputs=[smart_pending_qa, smart_txt_path_state],
        outputs=[save_qa_msg_smart],
    )

    smart_info_toggle.click(
        lambda visible: (gr.update(visible=not visible), not visible),
        inputs=[smart_info_visible],
        outputs=[smart_info_panel, smart_info_visible],
    )

    # 智能 Tab 圖片預覽
    smart_show_img_btn.click(
        toggle_doc_images,
        inputs=[smart_txt_path_state, smart_img_visible],
        outputs=[smart_img_group, smart_img_gallery, smart_img_visible],
    )

    # ── Embedding 快取 Tab 事件 ────────────────────────────────────────────

    def _delete_single_emb_cache(folder_name):
        if not folder_name:
            return "請先選擇資料夾", [], gr.update()
        folder_path = Path("./text_result") / folder_name
        txt_files = list(folder_path.glob("*.txt"))
        if not txt_files:
            return "❌ 找不到 txt 檔案", list_cached_collections(), gr.update()
        msg_result = delete_cache_for_txt(str(txt_files[0]))
        folders = get_history_folders()
        return msg_result, list_cached_collections(), gr.update(choices=folders)

    # 查看快取：同步刷新 JSON 清單 + 資料夾下拉選單
    refresh_cache_btn.click(
        lambda: (list_cached_collections(), gr.update(choices=get_history_folders())),
        inputs=[],
        outputs=[cache_list_display, cache_folder_dropdown],
    )

    # 刪除全部：刷新 JSON 清單 + 下拉選單
    delete_all_cache_btn.click(
        lambda: (delete_all_cache(), list_cached_collections(),
                 gr.update(choices=get_history_folders())),
        inputs=[],
        outputs=[cache_op_msg, cache_list_display, cache_folder_dropdown],
    )

    # 刪除單筆：刷新 JSON 清單 + 下拉選單
    delete_single_cache_btn.click(
        _delete_single_emb_cache,
        inputs=[cache_folder_dropdown],
        outputs=[cache_op_msg, cache_list_display, cache_folder_dropdown],
    )

    # ── QA 問答快取 Tab 事件 ──────────────────────────────────────────────

    qa_folder_dropdown.change(
        render_qa_html,
        inputs=[qa_folder_dropdown],
        outputs=[qa_entries_html],
    )

    # 重新整理：同步刷新下拉選單 + 當前選中資料夾的內容
    def _qa_refresh(folder):
        return gr.update(choices=get_history_folders()), render_qa_html(folder)

    qa_refresh_btn.click(
        _qa_refresh,
        inputs=[qa_folder_dropdown],
        outputs=[qa_folder_dropdown, qa_entries_html],
    )

    # 刪除此資料夾快取：刷新內容 + 下拉選單
    def _qa_delete_folder(folder):
        msg_result = delete_folder_qa_cache(folder)
        return msg_result, render_qa_html(folder), gr.update(choices=get_history_folders())

    qa_delete_folder_btn.click(
        _qa_delete_folder,
        inputs=[qa_folder_dropdown],
        outputs=[qa_op_msg, qa_entries_html, qa_folder_dropdown],
    )

    # 刪除所有快取：清空內容 + 刷新下拉選單
    def _qa_delete_all():
        msg_result = delete_all_qa_cache()
        return msg_result, "", gr.update(choices=get_history_folders())

    qa_delete_all_btn.click(
        _qa_delete_all,
        inputs=[],
        outputs=[qa_op_msg, qa_entries_html, qa_folder_dropdown],
    )

    def _delete_single_qa_entry(folder_name, entry_id):
        if not folder_name:
            return "❌ 請先選擇資料夾", ""
        if not entry_id or not entry_id.strip():
            return "❌ 請輸入要刪除的 ID", render_qa_html(folder_name)
        folder = Path("./text_result") / folder_name
        txt_files = list(folder.glob("*.txt"))
        if not txt_files:
            return "❌ 找不到 txt 檔案", render_qa_html(folder_name)
        result = delete_qa_entry_by_id(str(txt_files[0]), entry_id.strip())
        return result, render_qa_html(folder_name)

    qa_delete_entry_btn.click(
        _delete_single_qa_entry,
        inputs=[qa_folder_dropdown, qa_entry_id_input],
        outputs=[qa_op_msg, qa_entries_html],
    ).then(lambda: "", None, qa_entry_id_input)


if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=8080)

C:\Users\user\anaconda3\envs\contract\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
`torch_dtype` is deprecated! Use `dtype` instead!


正在載入 OCR 模型...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


✅ OCR 模型載入完成
正在載入 RAG 模型 (LLM + Embedding)...
✅ RAG 模型載入完成
* Running on local URL:  http://0.0.0.0:8080
* To create a public link, set `share=True` in `launch()`.


Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\user\anaconda3\envs\contract\Lib\asyncio\events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\user\anaconda3\envs\contract\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] 遠端主機已強制關閉一個現存的連線。
